In [2]:
import sys
sys.path.append("..")

from project.segmentation.medsam3 import MedSAM3Segmenter, MedSAM3Config
from project.data_io.reader import MedicalImageReader

KeyboardInterrupt: 

In [3]:
from dotenv import load_dotenv
load_dotenv('../.env')

True

In [ ]:
reader = MedicalImageReader()
image = reader.load("../data/raw/xray/1256842362861431725328351539259305635_u1qifz.png").resize((512, 512))

print(f"Image shape : {image.volume.shape}")
print(f"Image dtype : {image.volume.dtype}")
print(f"Source path : {image.source_path}")

Image shape : (1024, 1024, 3)
Image dtype : float32
Source path : ../data/raw/xray/1256842362861431725328351539259305635_u1qifz.png


In [5]:
import os
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
config = MedSAM3Config(
    prompts=["lung", "heart"],   # text prompts — change to match your image
    score_threshold=0.5,
    nms_iou=0.5,
    resolution=1008,
    device="cuda",
)

segmenter = MedSAM3Segmenter(config)
print("Segmenter ready.")

🔧 Initializing SAM3 + LoRA...
   Device: cuda
   Resolution: 1008x1008
   Confidence threshold: 0.5
   NMS IoU threshold: 0.5

📦 Building SAM3 model...


config.json: 0.00B [00:00, ?B/s]

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

🔗 Applying LoRA configuration...
Replaced 61 nn.MultiheadAttention modules with MultiheadAttentionLoRA
Applied LoRA to 458 modules:
  - backbone.vision_backbone.trunk.blocks.0.attn.qkv
  - backbone.vision_backbone.trunk.blocks.0.attn.proj
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.1.attn.qkv
  - backbone.vision_backbone.trunk.blocks.1.attn.proj
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.2.attn.qkv
  - backbone.vision_backbone.trunk.blocks.2.attn.proj
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.3.attn.qkv
  - backbone.vision_backbone.trunk.blocks.3.attn.proj
  - backbone.vision_backbone.trunk.blocks.3.mlp.fc1
  ... and 443 more
💾 Loading LoRA weights from /media/apoloml/DATOS_2/Tesis_

: 

In [ ]:
objects = segmenter.segment(image)
print(f"Objects detected: {len(objects)}")

for obj in objects:
    print(f"  id={obj.id}  confidence={obj.confidence:.3f}  mask_shape={obj.mask.shape}")